# MinneApple YOLOv8n-seg 기본 학습
로컬 검수 완료 데이터: 학습 518장 / 검증 152장 / 사과 주석 28,183개.

1. 상단 **런타임 → 런타임 유형 변경 → GPU**를 선택합니다.
2. 아래 셀을 위에서부터 실행합니다. Google Drive 접근 승인 화면은 직접 처리합니다.
3. 데이터는 VM으로 복사해 사용하고, 모델·학습 기록은 매 epoch 후 Drive로 백업합니다.

설정: 640 / batch 1 / mask_ratio 1 / 최대 50 epoch / patience 10 / 증강 없음.
로컬 OOM 시도는 보존하고 COCO 사전학습 가중치부터 새로 학습합니다.
배치·해상도·마스크 비율은 오류 발생 시 자동으로 변경하지 않습니다.
검증 수치는 모델 선택용이며 독립 시험 결과가 아닙니다.

데이터: [MinneApple](https://github.com/nicolaihaeni/MinneApple), Häni, Roy, Isler,
CC BY-NC-SA 3.0 US. 공개용 공유 권한은 변경하지 않습니다.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ultralytics==8.4.215"], check=True)
import torch, ultralytics
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
assert ultralytics.__version__ == "8.4.215"
print("torch:", torch.__version__, "ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


## Google Drive 연결
인증·접근 승인 화면은 본인이 직접 진행하세요. 이 셀은 승인까지 기다립니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 데이터 복사 및 무결성 확인
ZIP 파일을 Drive에서 VM 디스크로 복사합니다. 같은 분할과 라벨인지 SHA-256 및 객체 수를 검사합니다.

In [ ]:
from pathlib import Path
import hashlib, shutil, zipfile, importlib.util, json
DRIVE_PROJECT = Path("/content/drive/MyDrive") / 'apple-harvesting-robot-colab-20261116'
PARTS = [{'name': 'minneapple_colab_v1.zip.001', 'bytes': 419430400, 'sha256': '4a8346d995316d5d0c60adefa8cda5d402ad6353b541db3fd01c42b7da44f6b5'}, {'name': 'minneapple_colab_v1.zip.002', 'bytes': 419430400, 'sha256': 'd84dbff2b60f411ee5d56f9f153c95cbb7a39e60df7d1ad9400a1493c6405fa5'}, {'name': 'minneapple_colab_v1.zip.003', 'bytes': 407453432, 'sha256': '02271d783fa7375d1ea67efa6fb951e67e8d50a90cabf822468d0daf463e998b'}]
LOCAL_ARCHIVE = Path("/content/minneapple_colab_v1.zip")
with LOCAL_ARCHIVE.open("wb") as destination:
    for part in PARTS:
        source = DRIVE_PROJECT / part["name"]
        assert source.is_file(), f"Drive 업로드 확인: {source}"
        digest = hashlib.sha256()
        copied = 0
        with source.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                destination.write(block)
                digest.update(block)
                copied += len(block)
        assert copied == part["bytes"], f"전송 크기 불일치: {source.name}"
        assert digest.hexdigest() == part["sha256"], f"전송 체크섬 불일치: {source.name}"
        print("전송 확인:", source.name)
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
assert file_sha256(LOCAL_ARCHIVE) == '3c27555ed6642010b854db3e6e0ed5f83c8c0454a665ca3df23087cd72d2933e', "전송 파일 체크섬 불일치"
ROOT = Path("/content/apple_baseline_bundle")
if ROOT.exists():
    raise RuntimeError("이미 압축을 푼 런타임입니다. 검수 결과가 있으면 이 셀을 건너뛰세요.")
with zipfile.ZipFile(LOCAL_ARCHIVE) as archive:
    for member in archive.infolist():
        if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError("Invalid archive path")
    archive.extractall(ROOT)
spec = importlib.util.spec_from_file_location("apple_colab_runner", ROOT / "train_colab.py")
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
runner.verify_bundle(ROOT)
print("데이터 준비 완료:", ROOT)


## 기본 학습 실행
완료 전까지 이 셀은 실행 중으로 표시됩니다. 매 epoch의 DRIVE_BACKUP 메시지를 확인하세요.
연결이 끊기더라도 실행 VM이 살아 있을 수 있으므로, 기존 작업 상태를 확인한 뒤 재실행하세요.
오류가 발생하면 마지막 출력과 Drive의 baseline_summary.json을 확인하세요.


In [ ]:
result = runner.main(ROOT, DRIVE_PROJECT / "outputs")
print(json.dumps(result, ensure_ascii=False, indent=2))


## 결과 확인
학습 성공 여부, 완료 epoch, 저장 경로를 확인합니다. 새 이미지 추론은 다음 단계에서 진행합니다.

In [ ]:
from IPython.display import display, Image
assert result["status"] == "completed", result
print("완료 epoch:", result["epochs_completed"])
print("Drive 저장:", result["drive_backup"])
run_dir = Path(result["run"])
if (run_dir / "results.png").is_file():
    display(Image(filename=str(run_dir / "results.png")))
display(__import__("pandas").read_csv(run_dir / "results.csv").tail())
